# Error analysis and explainability of an image classifier

A classifier of landscape photos (6 classes: buildings, forest, glacier, mountain, sea, street) gets about one image in ten wrong. On which images, why, and should the model change or the data be fixed? The images come from the [Intel Image Classification](https://www.kaggle.com/datasets/puneet6060/intel-image-classification) dataset, 150 × 150 pixel photos.

1. A pretrained model
2. Training its last layer
3. The slices to work on first
4. The confusion matrix and the causes of the mistakes
5. Explaining mistakes with SHAP
6. Model or labels?

The notebook runs without a GPU, in about ten minutes: the pretrained network stays frozen, and its outputs are computed only once, on a subset of the training images.

## Data

The images sit in one folder per class: `landscape/seg_train` (14,034 images) and `landscape/seg_test` (3,000). `get_loader` loads them with the pretrained model's transforms, `WEIGHTS.transforms()` (resized to 224 × 224 and normalized as ImageNet). To take a few minutes without a GPU, the model trains on only `TRAIN_PER_CLASS` images per class, drawn at random; the test set is complete.

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh landscape

In [ ]:
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import PIL.Image
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, models

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
WEIGHTS = models.MobileNet_V3_Small_Weights.DEFAULT
TRAIN_PER_CLASS = 300  # training images per class


def get_loader(
  directory: str, shuffle: bool = False, per_class: int | None = None
) -> DataLoader:
  dataset = datasets.ImageFolder(directory, transform=WEIGHTS.transforms())
  if per_class is not None:  # per_class images drawn at random from each class
    random.seed(0)
    dataset.samples = [
      sample
      for target in range(len(dataset.classes))
      for sample in random.sample(
        [s for s in dataset.samples if s[1] == target], per_class
      )
    ]
    dataset.targets = [target for _, target in dataset.samples]
  return DataLoader(dataset, batch_size=64, shuffle=shuffle)


train_image_loader = get_loader("landscape/seg_train", per_class=TRAIN_PER_CLASS)
test_image_loader = get_loader("landscape/seg_test")
CLASSES = test_image_loader.dataset.classes
train_paths = [path for path, _ in train_image_loader.dataset.samples]
test_paths = [path for path, _ in test_image_loader.dataset.samples]
print(f"Classes: {CLASSES}")
print(f"{len(train_paths)} training images, {len(test_paths)} test images")

## 1. A pretrained model

We start from a MobileNetV3 *small* pretrained on ImageNet (the `WEIGHTS` weights). Its convolutional part, `model.features`, is frozen (`requires_grad_(False)`: no more gradient for its parameters), and its last layer, `model.classifier[-1]`, which output ImageNet's 1,000 classes, is replaced with a linear layer with one output per class of `CLASSES`. Only the layers of `model.classifier` remain trainable.

In [ ]:
model = models.mobilenet_v3_small(weights=WEIGHTS)
model.features.requires_grad_(False)
model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, len(CLASSES))
model = model.to(DEVICE)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"{trainable:,} trainable parameters out of {total:,}")

## 2. Training the last layer

The frozen part always returns the same 576-feature vector for a given image: `extract_features` computes it once and for all, for each image (a few minutes without a GPU). The layers of `model.classifier` then train on these vectors, in a few seconds, with the usual `train_epoch`, `evaluate` and `predict` functions.

In [ ]:
@torch.inference_mode()
def extract_features(model: nn.Module, loader: DataLoader) -> TensorDataset:
  """The frozen part's outputs (features then avgpool), and the labels."""
  model.eval()
  features, labels = [], []
  for images, targets in loader:
    features.append(model.avgpool(model.features(images.to(DEVICE))).flatten(1).cpu())
    labels.append(targets)
  return TensorDataset(torch.cat(features), torch.cat(labels))


train_features = extract_features(model, train_image_loader)
test_features = extract_features(model, test_image_loader)
print(f"Test features: {tuple(test_features.tensors[0].shape)}")

In [ ]:
def train_epoch(model: nn.Module, loader: DataLoader, optimizer) -> float:
  """One epoch; returns the mean loss."""
  model.train()
  loss_fn = nn.CrossEntropyLoss()
  total_loss = 0.0
  for x, y in loader:
    x, y = x.to(DEVICE), y.to(DEVICE)
    optimizer.zero_grad()
    loss = loss_fn(model(x), y)
    loss.backward()
    optimizer.step()
    total_loss += loss.item() * len(x)
  return total_loss / len(loader.dataset)


@torch.inference_mode()
def evaluate(model: nn.Module, loader: DataLoader) -> float:
  """Accuracy on loader."""
  model.eval()
  correct = 0
  for x, y in loader:
    correct += (model(x.to(DEVICE)).argmax(dim=1).cpu() == y).sum().item()
  return correct / len(loader.dataset)


@torch.inference_mode()
def predict(model: nn.Module, loader: DataLoader) -> torch.Tensor:
  """Indices of the predicted classes."""
  model.eval()
  return torch.cat([model(x.to(DEVICE)).argmax(dim=1).cpu() for x, _ in loader])


torch.manual_seed(0)
train_loader = DataLoader(train_features, batch_size=64, shuffle=True)
test_loader = DataLoader(test_features, batch_size=64)
optimizer = torch.optim.AdamW(model.classifier.parameters(), lr=0.001)
for epoch in range(10):
  loss = train_epoch(model.classifier, train_loader, optimizer)
  print(f"Epoch {epoch + 1}: loss {loss:.3f}")
print(f"Test set accuracy: {evaluate(model.classifier, test_loader):.3f}")

## 3. The slices to work on first

The classes are natural slices of the data: the method of the frame "Recommended Approach --- Example, End" tells which one to work on first. Compute `slices`, a `DataFrame` indexed by `CLASSES`, with, for each class, its accuracy on the test images (the share of its images classified right) and its proportion among the test images. `REFERENCE` gives the per-class accuracy of a stronger model, the same network trained on the 14,034 training images (this lab's demo): lacking a measured human-level performance, it's the reference for what is achievable. Add each class's potential impact, (reference − accuracy) × proportion, and sort the classes by decreasing impact. Which slice should be worked on first? Is the ranking by impact the same as by accuracy alone?

**Useful documentation**

- [`numpy.bincount`](https://numpy.org/doc/stable/reference/generated/numpy.bincount.html)
- [`pandas.DataFrame`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html)
- [`DataFrame.sort_values`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sort_values.html)

In [ ]:
test_labels = test_features.tensors[1].numpy()
test_preds = predict(model.classifier, test_loader).numpy()
# Per-class accuracy of the same network trained on the 14,034 images
REFERENCE = {
  "buildings": 0.957,
  "forest": 0.996,
  "glacier": 0.899,
  "mountain": 0.838,
  "sea": 0.967,
  "street": 0.902,
}

# Your code here

### Additional hints

The calls and parameters to use:

- `test_labels` and `test_preds`: the true classes of the test images and the predicted classes, in the order of `test_paths`
- `(test_preds[test_labels == c] == c).mean()`: the accuracy of the class with index `c`, the share of its images predicted `c`
- `np.bincount(test_labels) / len(test_labels)`: each class's proportion, in the order of `CLASSES`
- `pd.DataFrame({"accuracy": ..., "proportion": ...}, index=CLASSES)`: one row per class; `pd.Series(REFERENCE)`, added as a column, aligns on this index
- `slices.sort_values("impact", ascending=False)`: the largest impacts first
- To check: the proportions add up to 1; a negative impact means the model does better than the reference on that class

A skeleton to copy into the exercise cell, replacing each `...`:

```python
accuracy = [(test_preds[test_labels == c] == c).mean() for c in range(len(CLASSES))]
slices = pd.DataFrame(
  {"accuracy": ..., "proportion": np.bincount(...) / len(test_labels)},
  index=CLASSES,
)
slices["reference"] = pd.Series(REFERENCE)
slices["impact"] = (... - slices["accuracy"]) * ...
slices.sort_values(..., ascending=False).round(3)
```

### Solution

In [ ]:
accuracy = [(test_preds[test_labels == c] == c).mean() for c in range(len(CLASSES))]
slices = pd.DataFrame(
  {"accuracy": accuracy, "proportion": np.bincount(test_labels) / len(test_labels)},
  index=CLASSES,
)
slices["reference"] = pd.Series(REFERENCE)
slices["impact"] = (slices["reference"] - slices["accuracy"]) * slices["proportion"]
slices.sort_values("impact", ascending=False).round(3)

Answer elements:

- On our run, "glacier" has the largest impact (1.4% of the test images): it's the slice to work on first. "sea" and "buildings" come next, far behind.
- The ranking isn't the one by accuracy: "mountain" has the second lowest accuracy (0.87), but a negative impact. The stronger model doesn't do better on this class: there is nothing to gain there by imitating it. Accuracy tells where the model is wrong, impact where it can improve.
- "glacier" and "mountain" are read together: this model puts more images in "mountain" than the reference model, which raises the accuracy of "mountain" and lowers that of "glacier". The confusion matrix, in the next part, shows it.
- The reference isn't human-level performance: on this pair, a human would hesitate too (parts 5 and 6), and the achievable gain may be smaller.
- Your numbers may differ a little: even without a GPU, training doesn't give exactly the same model from one run to the next.

## 4. The confusion matrix and the causes of the mistakes

Show the confusion matrix of the test images, from `test_labels` and `test_preds`. Then compute `pairs`, the three most frequent (true class, predicted class) pairs among the mistakes, with their number of images. Finally, show about ten images of the most frequent pair with `show_images`. Do these mistakes look alike? Can the pair's two classes always be told apart by eye?

**Useful documentation**

- [`ConfusionMatrixDisplay.from_predictions`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.ConfusionMatrixDisplay.html#sklearn.metrics.ConfusionMatrixDisplay.from_predictions)
- [`confusion_matrix`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html)
- [`numpy.flatnonzero`](https://numpy.org/doc/stable/reference/generated/numpy.flatnonzero.html)

In [ ]:
def show_images(paths: list[str], titles: list[str]) -> None:
  """Shows images side by side, with their titles."""
  _, axes = plt.subplots(1, len(paths), figsize=(2 * len(paths), 2.4), squeeze=False)
  for ax, path, title in zip(axes[0], paths, titles):
    ax.imshow(PIL.Image.open(path))
    ax.set_title(title, fontsize=9)
    ax.axis("off")
  plt.show()


# Your code here

### Additional hints

The calls and parameters to use:

- `ConfusionMatrixDisplay.from_predictions(test_labels, test_preds, display_labels=CLASSES, xticks_rotation=45)`: true classes as rows, predicted ones as columns
- `confusion_matrix(test_labels, test_preds)[true, predicted]`: the number of images of class `true` predicted `predicted`; the mistakes are off the diagonal (`true != predicted`)
- `sorted(..., reverse=True)[:3]` on `(count, true class, predicted class)` triples: the three largest counts first
- `np.flatnonzero((test_labels == a) & (test_preds == b))`: the positions of the images of class `a` predicted `b`, which are also their positions in `test_paths`

A skeleton to copy into the exercise cell, replacing each `...`:

```python
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

ConfusionMatrixDisplay.from_predictions(
  ..., ..., display_labels=CLASSES, xticks_rotation=45
)
plt.show()

matrix = confusion_matrix(test_labels, test_preds)
pairs = sorted(
  (
    (matrix[true, predicted], CLASSES[true], CLASSES[predicted])
    for true in range(len(CLASSES))
    for predicted in range(len(CLASSES))
    if ...
  ),
  reverse=True,
)[:3]
for count, true, predicted in pairs:
  print(f'{count} "{true}" images predicted "{predicted}"')

count, true, predicted = pairs[0]
mistakes = np.flatnonzero(
  (test_labels == CLASSES.index(true)) & (... == CLASSES.index(...))
)
show_images([test_paths[i] for i in mistakes[:10]], [f"no. {i}" for i in mistakes[:10]])
```

### Solution

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

ConfusionMatrixDisplay.from_predictions(
  test_labels, test_preds, display_labels=CLASSES, xticks_rotation=45
)
plt.show()

matrix = confusion_matrix(test_labels, test_preds)
pairs = sorted(
  (
    (matrix[true, predicted], CLASSES[true], CLASSES[predicted])
    for true in range(len(CLASSES))
    for predicted in range(len(CLASSES))
    if true != predicted
  ),
  reverse=True,
)[:3]
for count, true, predicted in pairs:
  print(f'{count} "{true}" images predicted "{predicted}"')

count, true, predicted = pairs[0]
mistakes = np.flatnonzero(
  (test_labels == CLASSES.index(true)) & (test_preds == CLASSES.index(predicted))
)
show_images([test_paths[i] for i in mistakes[:10]], [f"no. {i}" for i in mistakes[:10]])

### The causes of the mistakes

The matrix tells which classes are mistaken for one another, not why. As in the frame "Finding the Slices", examine the first 20 mistakes of the most frequent pair, `reviewed`, and give each one a cause label: "snowy mountain" (peaks and snow or ice at once: an ambiguous case, where a human would hesitate too), "wrong label" (the image isn't of its class: no ice on a "glacier", for instance), "atypical image" (an aerial view, a drawing, an unusual framing), or "other" (a clear image of its class, which the model misses). Fill in `causes`, a dictionary {image number: cause label}, then count the mistakes per cause label with a `Counter`. Which cause dominates? Which slice should be measured next, and what should be done about the mistakes of each cause?

**Useful documentation**

- [`collections.Counter`](https://docs.python.org/3/library/collections.html#collections.Counter)

In [ ]:
from collections import Counter

CAUSES = ["snowy mountain", "wrong label", "atypical image", "other"]
reviewed = mistakes[:20]
for row in (reviewed[:10], reviewed[10:]):
  show_images([test_paths[i] for i in row], [f"no. {i}" for i in row])

causes = {}  # {image number: one of the CAUSES}

# Your code here

#### Additional hints

The calls and parameters to use:

- The keys of `causes` are the numbers shown above the images, the values strings of `CAUSES`: `{924: "snowy mountain", 966: "other", ...}`
- One label per image, the main cause; the frame allows several per mistake: a dictionary of lists and a `Counter` over all the labels would allow it
- `Counter(causes.values())` counts the labels; `.most_common()` returns them from the most to the least frequent, as (label, count) pairs

A skeleton to copy into the exercise cell, replacing each `...`:

```python
causes = {
  ...: "snowy mountain",
  ...: "wrong label",
  # ...
}
counts = Counter(...)
for cause, count in counts.most_common():
  print(f"{cause}: {count} ({count / len(causes):.0%})")
```

#### Solution

In [ ]:
causes = {
  922: "atypical image",
  924: "snowy mountain",
  938: "wrong label",
  943: "snowy mountain",
  950: "wrong label",
  956: "wrong label",
  962: "snowy mountain",
  966: "other",
  976: "snowy mountain",
  994: "snowy mountain",
  999: "snowy mountain",
  1001: "other",
  1020: "other",
  1030: "other",
  1033: "atypical image",
  1034: "atypical image",
  1043: "atypical image",
  1049: "wrong label",
  1050: "other",
  1054: "snowy mountain",
}
counts = Counter(causes.values())
for cause, count in counts.most_common():
  print(f"{cause}: {count} ({count / len(causes):.0%})")

Answer elements (on our run, glaciers predicted "mountain"; your mistakes may differ a little):

- "snowy mountain" dominates (7 out of 20, 35%): peaks and ice in the same image, whether the label is "glacier" or "mountain". It's the "Snowy mountains" slice of the frame "Recommended Approach --- Example": it's measured on its own next (the model's accuracy, and human-level performance on a relabeled sample), and it's handled through the labeling guide, not through the model.
- "other" (5): unambiguous glaciers (seracs, ice fronts) that the model misses. There, the model is wrong: more training images (the reference model, trained on all the images, does better on "glacier"), or data augmentation.
- "wrong label" (4): lakes, a forest, green mountains, labeled "glacier". They are relabeled; no model can get them "right".
- "atypical image" (4): aerial views, an iceberg beach, rare in the data. Add similar ones, or accept these mistakes knowingly.
- Twenty mistakes give an order of magnitude; the frame advises examining about a hundred before deciding.

## 5. Explaining mistakes with SHAP

Mountains mistaken for glaciers are among the most frequent mistakes. To see what pushes the model toward "glacier", explain six of them with SHAP, those of `images`. First write `predict_proba(images)`, which takes a batch of NumPy images of shape (N, 150, 150, 3), with values between 0 and 1, and returns the probabilities of the 6 classes, of shape (N, 6): SHAP calls the model only through this function. Then create an explainer that masks areas of the images by replacing them with gray, compute `explanation` for the "glacier" and "mountain" outputs, and show it. Which areas of the images push the model toward "glacier"? Toward "mountain"?

**Useful documentation**

- [`Tensor.permute`](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.permute.html)
- [`shap.Explainer`](https://shap.readthedocs.io/en/latest/generated/shap.Explainer.html)
- [`shap.maskers.Image`](https://shap.readthedocs.io/en/latest/generated/shap.maskers.Image.html)
- [`shap.plots.image`](https://shap.readthedocs.io/en/latest/generated/shap.plots.image.html)

In [ ]:
GLACIER, MOUNTAIN = CLASSES.index("glacier"), CLASSES.index("mountain")
mountains_as_glaciers = np.flatnonzero(
  (test_labels == MOUNTAIN) & (test_preds == GLACIER)
)
print(f"{len(mountains_as_glaciers)} mountains predicted glacier")


def load_image(path: str) -> np.ndarray:
  """An image as a (150, 150, 3) array, with values between 0 and 1."""
  image = PIL.Image.open(path).convert("RGB").resize((150, 150))
  return np.asarray(image, dtype=np.float32) / 255


images = np.stack([load_image(test_paths[i]) for i in mountains_as_glaciers[:6]])


def predict_proba(images: np.ndarray) -> np.ndarray: ...


# Your code here

### Additional hints

The calls and parameters to use:

- `torch.as_tensor(images, dtype=torch.float32).permute(0, 3, 1, 2)`: from a NumPy batch (N, height, width, channels) to a PyTorch tensor (N, channels, height, width)
- `WEIGHTS.transforms()(batch)`: the same resizing and normalization as during training
- `model(...).softmax(dim=1)`, under `torch.inference_mode()` after `model.eval()`, then `.cpu().numpy()`
- To check: each row of `predict_proba(images)` has its largest probability in the `GLACIER` column (2), and sums to 1
- `shap.maskers.Image(0.5, images[0].shape)`: the masked areas are replaced with gray (0.5); the `"blur(32,32)"` masker of shap's examples needs OpenCV
- `shap.Explainer(predict_proba, masker, output_names=CLASSES)`: with an image masker, shap picks the *Partition* explainer, which splits the image into smaller and smaller blocks
- `explainer(images, max_evals=500, batch_size=50, outputs=[GLACIER, MOUNTAIN])`: 500 calls to the model per image (about fifteen seconds per image without a GPU), in batches of 50; `outputs`: the classes to explain
- `shap.plots.image(explanation)`: in red, the areas that push toward the column's class; in blue, those that push away from it

A skeleton to copy into the exercise cell, replacing each `...`:

```python
import shap


@torch.inference_mode()
def predict_proba(images: np.ndarray) -> np.ndarray:
  model.eval()
  batch = torch.as_tensor(images, dtype=torch.float32).permute(...)
  return model(WEIGHTS.transforms()(batch).to(DEVICE)).softmax(dim=...).cpu().numpy()


print(predict_proba(images).round(2))

masker = shap.maskers.Image(0.5, images[0].shape)
explainer = shap.Explainer(..., masker, output_names=CLASSES)
explanation = explainer(images, max_evals=500, batch_size=50, outputs=[..., ...])
shap.plots.image(...)
```

### Solution

In [ ]:
import shap


@torch.inference_mode()
def predict_proba(images: np.ndarray) -> np.ndarray:
  model.eval()
  batch = torch.as_tensor(images, dtype=torch.float32).permute(0, 3, 1, 2)
  return model(WEIGHTS.transforms()(batch).to(DEVICE)).softmax(dim=1).cpu().numpy()


print(predict_proba(images).round(2))

masker = shap.maskers.Image(0.5, images[0].shape)
explainer = shap.Explainer(predict_proba, masker, output_names=CLASSES)
explanation = explainer(
  images, max_evals=500, batch_size=50, outputs=[GLACIER, MOUNTAIN]
)
shap.plots.image(explanation)

## 6. Model or labels?

A mistake comes from the model when the image is clear but the model relies on a wrong cue. It comes from the labels when similar images carry sometimes one label, sometimes the other: then no model can learn the boundary. To decide, write `neighbours(indices, k)`, which returns, for each test image of `indices`, the positions in `train_paths` of the `k` closest training images, in the frozen network's feature space (the vectors of `test_features` and `train_features`, compared by cosine similarity). Compare the share of "glacier" labels among the 5 neighbors of the mountains predicted glacier, and among those of the mountains classified right. Finally, show the first three mistakes of `mountains_as_glaciers` and their neighbors, with their labels. Judging from these neighbors and the SHAP figures, do these mistakes come from the model or from the labels? What should be done?

**Useful documentation**

- [`torch.nn.functional.normalize`](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.normalize.html)
- [`torch.topk`](https://docs.pytorch.org/docs/stable/generated/torch.topk.html)

In [ ]:
def neighbours(indices: np.ndarray, k: int = 5) -> np.ndarray: ...


# Your code here

### Additional hints

The calls and parameters to use:

- `normalize(vectors, dim=1)` (from `torch.nn.functional`): each vector scaled to a norm of 1; the dot product of two normalized vectors is their cosine similarity
- `test_vectors[indices] @ train_vectors.T`: the similarity of each test image of `indices` with each training image, of shape (len(indices), 1,800)
- `similarities.topk(k, dim=1).indices`: the positions of the `k` largest similarities of each row; they are positions in `train_paths` and in `train_features`
- `train_labels[neighbours(indices)]`: the neighbors' labels, of shape (len(indices), k); `(... == GLACIER).mean()`: the share of "glacier"
- To check: `neighbours(mountains_as_glaciers[:2]).shape` is `(2, 5)`

A skeleton to copy into the exercise cell, replacing each `...`:

```python
from torch.nn.functional import normalize

train_vectors = normalize(train_features.tensors[0], dim=1)
test_vectors = normalize(..., dim=1)
train_labels = train_features.tensors[1].numpy()


def neighbours(indices: np.ndarray, k: int = 5) -> np.ndarray:
  similarities = test_vectors[indices] @ train_vectors.T
  return similarities.topk(..., dim=1).indices.numpy()


well_classified = np.flatnonzero((test_labels == MOUNTAIN) & (test_preds == ...))
for name, indices in [
  ("Mountains predicted glacier", mountains_as_glaciers),
  ("Mountains classified right", well_classified),
]:
  share = (train_labels[neighbours(indices)] == ...).mean()
  print(f'{name}: {share:.0%} of neighbors labeled "glacier"')

for i, near in zip(mountains_as_glaciers[:3], neighbours(mountains_as_glaciers[:3])):
  show_images(
    [test_paths[i]] + [train_paths[j] for j in near],
    [f"test no. {i}"] + [CLASSES[...] for j in near],
  )
```

### Solution

In [ ]:
from torch.nn.functional import normalize

train_vectors = normalize(train_features.tensors[0], dim=1)
test_vectors = normalize(test_features.tensors[0], dim=1)
train_labels = train_features.tensors[1].numpy()


def neighbours(indices: np.ndarray, k: int = 5) -> np.ndarray:
  similarities = test_vectors[indices] @ train_vectors.T
  return similarities.topk(k, dim=1).indices.numpy()


well_classified = np.flatnonzero((test_labels == MOUNTAIN) & (test_preds == MOUNTAIN))
for name, indices in [
  ("Mountains predicted glacier", mountains_as_glaciers),
  ("Mountains classified right", well_classified),
]:
  share = (train_labels[neighbours(indices)] == GLACIER).mean()
  print(f'{name}: {share:.0%} of neighbors labeled "glacier"')

for i, near in zip(mountains_as_glaciers[:3], neighbours(mountains_as_glaciers[:3])):
  show_images(
    [test_paths[i]] + [train_paths[j] for j in near],
    [f"test no. {i}"] + [CLASSES[train_labels[j]] for j in near],
  )

Answer elements:

- The mountains predicted glacier have far more "glacier" neighbors than the mountains classified right: in the training set, very similar photos are labeled "glacier". The model reproduces the labels' ambiguity. The reverse pair, glaciers predicted mountain, is in fact one of the most frequent mistakes too.
- The SHAP figures point the same way: the red of "glacier" falls on snow and ice (snowy peaks, snowfields, ice fronts), sensible cues for this class, and not on an unrelated detail. Several of these "mountains" do show a glacier: a human would hesitate too, and some labels are wrong.
- So these mistakes mostly come from the labels. They are fixed in the data: make the labeling guide more precise (for instance "glacier as soon as ice or snow covers most of the image"), relabel the ambiguous cases, or allow several labels per image. A better model would gain almost nothing: on this slice, human-level performance is itself low.
- The mistake would come from the model if its neighbors were mountains and SHAP pointed to an unrelated cue (the sky, an edge of the image): then more varied examples, data augmentation or a better model would help.
- Limits: SHAP's blocks are coarse, and six images are not enough to conclude; examine more of them before relabeling.

## Key takeaways

- Error analysis starts with the confusion matrix: the most mistaken pairs of classes, computed rather than assumed, tell which mistakes to look at first.
- SHAP explains an image's prediction by areas: it masks blocks of the image and measures the effect on each class's probability. Its blocks are coarse, and it explains the model, not reality.
- Before changing the model, look at the labels: similar images labeled differently are fixed in the labeling guide and by relabeling, not with a better network.